In [ ]:
!pip install mediapipe

In [ ]:
import cv2
import os
import csv
import math
import mediapipe as mp

In [ ]:
mp_face_mesh = mp.solutions.face_mesh

def euclidean(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def measure_eye_and_face(image_path, draw=False):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        refine_landmarks=True,  # better eye landmarks
        max_num_faces=1,
        min_detection_confidence=0.6
    ) as fm:
        results = fm.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if not results.multi_face_landmarks:
            raise RuntimeError("No face detected.")

        lm = results.multi_face_landmarks[0].landmark
        def pt(i): return (int(lm[i].x * w), int(lm[i].y * h))

        # Landmarks
        R_outer, R_inner = pt(33), pt(133)   # right eye corners
        L_outer, L_inner = pt(263), pt(362)  # left eye corners
        top_face, chin = pt(10), pt(152)     # forehead to chin

        # Measurements
        right_eye_width = euclidean(R_outer, R_inner)
        left_eye_width = euclidean(L_outer, L_inner)
        face_height = euclidean(top_face, chin)

        return right_eye_width, left_eye_width, face_height

def extract_features_from_folder(folder_path, output_csv="features.csv"):
    results = []

    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(folder_path, file_name)
            try:
                features = measure_eye_and_face(image_path)
                results.append([file_name] + list(features))
                print(f"Processed {file_name} → {features}")
            except Exception as e:
                print(f"Skipping {file_name}: {e}")

    # Save results to CSV
    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Image", "Right Eye Width", "Left Eye Width", "Face Height"])
        writer.writerows(results)

    print(f"\n✅ Features saved to {output_csv}")

In [8]:
extract_features_from_folder("/content/drive/MyDrive/Soft Computing/Labsheet_01/kg")

Processed IMG-20250819-WA0031.jpg → (107.29864864013899, 131.86735759845953, 651.5120873782773)
Processed IMG-20250819-WA0032.jpg → (68.26419266350405, 76.6550715869472, 403.44764220404113)
Processed IMG-20250819-WA0055 - Copy - Copy.jpg → (30.265491900843113, 39.01281840626232, 192.26284092356485)
Processed IMG-20250819-WA0034.jpg → (52.773099207835045, 44.28317965096906, 270.266535109325)
Skipping IMG-20250819-WA0060.jpg: No face detected.
Processed IMG-20250819-WA0039 - Copy.jpg → (117.06835610018618, 135.1628647225265, 668.5544106503224)
Processed IMG-20250819-WA0032 - Copy.jpg → (68.26419266350405, 76.6550715869472, 403.44764220404113)
Skipping IMG-20250819-WA0060 - Copy.jpg: No face detected.
Processed IMG-20250819-WA0049.jpg → (16.0, 22.561028345356956, 134.18271125595876)
Processed IMG-20250819-WA0042.jpg → (53.36665625650534, 60.03332407921454, 305.62885989382613)
Processed IMG-20250819-WA0040 - Copy.jpg → (53.45091205957107, 65.37583651472461, 325.16149833582693)
Skipping IMG

In [9]:
extract_features_from_folder("/content/drive/MyDrive/Soft Computing/Labsheet_01/SC_Photos_Compressed")

Processed IMG_20250819_161209730.jpg → (243.00205760445732, 209.61870145576228, 1296.3892162464172)
Processed IMG_20250819_161155783.jpg → (233.13729860320507, 225.57482128996577, 1323.315910884472)
Processed IMG_20250819_161320227.jpg → (222.50617968946392, 230.471256342304, 1325.275065788231)
Processed IMG_20250819_161103890.jpg → (204.00980368599937, 206.77040407176264, 1192.4030358901305)
Processed IMG_20250819_160845147.jpg → (194.0103090044444, 197.1623696347759, 1144.0279716860073)
Processed IMG_20250819_161431341.jpg → (205.35335400231475, 211.62466774929618, 1095.1479352124077)
Processed IMG_20250819_161037857.jpg → (279.1146717748818, 282.70479302622374, 1624.3558723383248)
Processed IMG_20250819_160735106.jpg → (262.2746651889961, 263.4919353604584, 1545.3313560528047)
Processed IMG_20250819_161310907.jpg → (229.36869882353173, 229.36651891677653, 1337.063199702991)
Processed IMG_20250819_161044689.jpg → (295.3269374777723, 294.49108645254444, 1631.5894091345408)
Processed I